# Inference Parameters and Text Generation

Inference tuning is one of the most interview-tested LLM engineering topics because it directly affects user experience, hallucination rate, latency, and cost without changing model weights. Once a model emits next-token logits, decoding strategy and sampling parameters determine whether output is deterministic, diverse, repetitive, concise, or unstable.

This notebook provides an interview-prep deep dive into temperature, top-p, top-k, max tokens, repetition penalties, greedy/beam/sampling trade-offs, reproducibility with seeds, and production tuning patterns. The examples are self-contained and use only numpy/matplotlib so you can reason about decoding behavior from first principles.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

---

## 1. From Logits to Tokens

A model outputs logits \(z\), then sampling converts them to probabilities with softmax:

\[
P_i = \frac{\exp(z_i)}{\sum_j \exp(z_j)}
\]

Decoding rules (greedy, top-k, top-p) decide how to select the next token from this distribution.

---

## 2. Temperature

Temperature rescales logits before softmax:

\[
P_i(T) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}
\]

Lower \(T\) sharpens distribution and increases determinism. Higher \(T\) flattens distribution and increases diversity.

In [ ]:
# Expanded temperature demo inspired by a simple sampling script
vocab = np.array(["the", "cat", "sat", "on", "mat", "dog", "ran"])
base_logits = np.array([2.5, 2.0, 1.6, 1.2, 1.1, 1.0, 0.7])

def softmax(logits):
    z = logits - logits.max()
    e = np.exp(z)
    return e / e.sum()

def probs_with_temperature(logits, temperature):
    t = max(temperature, 1e-6)
    return softmax(logits / t)

temps = [0.2, 0.7, 1.0, 1.5]
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharey=True)
for ax, t in zip(axes.flat, temps):
    p = probs_with_temperature(base_logits, t)
    ax.bar(vocab, p)
    ax.set_title(f"temperature={t}")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

---

## 3. Greedy Decoding vs Sampling

Greedy decoding always picks the maximum-probability token. It is stable but can become repetitive. Sampling introduces controlled randomness and can produce more varied outputs, especially for open-ended generation.

---

## 4. Top-k Sampling

Top-k keeps only the \(k\) highest-probability tokens and renormalizes. This bounds randomness by excluding low-probability tails.

In [ ]:
def sample_top_k(probs, k, rng):
    idx = np.argsort(probs)[::-1][:k]
    p = probs[idx]
    p = p / p.sum()
    return rng.choice(idx, p=p)

rng = np.random.default_rng(0)
p = probs_with_temperature(base_logits, 1.0)
for k in [1, 3, 5]:
    picks = [vocab[sample_top_k(p, k, rng)] for _ in range(12)]
    print(f"top_k={k}: {picks}")

---

## 5. Top-p (Nucleus) Sampling

Top-p selects the smallest set of tokens whose cumulative probability exceeds threshold \(p\), then samples within that set. This makes candidate set size adaptive to distribution shape.

In [ ]:
def sample_top_p(probs, p_threshold, rng):
    order = np.argsort(probs)[::-1]
    sorted_probs = probs[order]
    cdf = np.cumsum(sorted_probs)
    cutoff = np.searchsorted(cdf, p_threshold) + 1
    keep = order[:cutoff]
    renorm = probs[keep] / probs[keep].sum()
    return rng.choice(keep, p=renorm)

rng = np.random.default_rng(1)
for pth in [0.5, 0.8, 0.95]:
    picks = [vocab[sample_top_p(p, pth, rng)] for _ in range(12)]
    print(f"top_p={pth}: {picks}")

---

## 6. Max Tokens and Stop Sequences

`max_tokens` limits generation length and protects latency budgets. Stop sequences terminate decoding when specific substrings appear, helping structure outputs for downstream systems.

---

## 7. Frequency and Presence Penalties

Penalties reduce repetition. A frequency penalty discourages repeatedly using the same token proportional to count. A presence penalty discourages any token that has appeared at least once. Both operate by adjusting logits before sampling.

In [ ]:
def apply_penalties(logits, token_counts, alpha_freq=0.0, alpha_pres=0.0):
    adjusted = logits.copy().astype(float)
    for i, c in enumerate(token_counts):
        adjusted[i] -= alpha_freq * c
        adjusted[i] -= alpha_pres * (1 if c > 0 else 0)
    return adjusted

counts = np.array([5, 1, 0, 2, 0, 0, 1])
adj = apply_penalties(base_logits, counts, alpha_freq=0.15, alpha_pres=0.25)
print("Original logits: ", base_logits)
print("Adjusted logits: ", np.round(adj, 3))
print("Original probs:  ", np.round(softmax(base_logits), 3))
print("Adjusted probs:  ", np.round(softmax(adj), 3))

---

## 8. Reproducibility and Seeds

Sampling is stochastic. Fixed seeds allow deterministic replay under the same model, prompt, and decoding settings. In production, reproducibility is useful for debugging and regression testing.

In [ ]:
def draw_sequence(seed):
    rng = np.random.default_rng(seed)
    probs = probs_with_temperature(base_logits, 0.9)
    return [vocab[rng.choice(len(vocab), p=probs)] for _ in range(8)]

print("seed=123:", draw_sequence(123))
print("seed=123:", draw_sequence(123))
print("seed=999:", draw_sequence(999))

---

## 9. Numpy Generation Loop (Pseudocode to Code)

A practical decoding loop repeatedly applies model logits, decoding rules, and stop conditions. The following toy implementation uses a transition matrix as the model and executes a full generation loop.

In [ ]:
toy_vocab = np.array(["I", "like", "to", "learn", "LLMs", ".", "<END>"])
tid = {t: i for i, t in enumerate(toy_vocab)}
T = np.array([
    [0.02, 0.50, 0.25, 0.08, 0.10, 0.03, 0.02],
    [0.02, 0.05, 0.40, 0.20, 0.25, 0.05, 0.03],
    [0.03, 0.05, 0.04, 0.45, 0.35, 0.05, 0.03],
    [0.03, 0.06, 0.08, 0.05, 0.50, 0.20, 0.08],
    [0.02, 0.04, 0.05, 0.12, 0.04, 0.58, 0.15],
    [0.25, 0.22, 0.12, 0.10, 0.08, 0.03, 0.20],
    [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 1.00],
])

def generate(prompt_token="I", max_tokens=15, temperature=1.0, top_k=4, stop_token="<END>", seed=3):
    rng = np.random.default_rng(seed)
    out = [prompt_token]
    counts = Counter([prompt_token])
    current = prompt_token

    for _ in range(max_tokens):
        probs = T[tid[current]]
        logits = np.log(np.maximum(probs, 1e-12))
        logits = apply_penalties(logits, np.array([counts[t] for t in toy_vocab]), alpha_freq=0.10, alpha_pres=0.05)
        step_probs = probs_with_temperature(logits, temperature)
        nxt_id = sample_top_k(step_probs, top_k, rng)
        nxt = toy_vocab[nxt_id]
        out.append(nxt)
        counts[nxt] += 1
        current = nxt
        if nxt in {stop_token, "."}:
            break

    return " ".join(out)

print(generate())
print(generate(seed=11, temperature=0.7))

---

## 10. Practical Tuning Strategy

A common sequence is: start deterministic (low temperature, modest top-k), verify task correctness, then gradually increase diversity. Add penalties only when repetition appears. Keep hard safety constraints through max token limits and explicit stop sequences.

---

## Summary

Inference quality depends on both model capability and decoding policy. Temperature, top-k, top-p, token limits, and penalties shape how logits become language. Treat these parameters as controllable levers: tune for correctness first, then for style and diversity under stable latency and budget constraints.

---

## 11. Mathematical View of Decoding Controls

Decoding applies a transformation pipeline to logits \(z\): temperature scaling, optional penalties, truncation (top-k or top-p), and stochastic/deterministic token selection. One useful decomposition is:

\[
\tilde{z}_i = \frac{z_i - \alpha_f c_i - \alpha_p \mathbf{1}(c_i>0)}{T}
\]

\[
P_i = \text{Softmax}(\tilde{z}_i), \quad \hat{P}_i = \text{Truncate}(P_i; k, p)
\]

where \(c_i\) is prior token count, \(\alpha_f\) is frequency penalty, and \(\alpha_p\) is presence penalty. This formula gives a clean interview explanation of how multiple knobs combine to shape the next-token distribution.

---

## 12. Greedy vs Beam Search vs Sampling

Greedy decoding chooses the local argmax at each step. It is fast and stable but can miss globally better sequences. Beam search tracks the top \(B\) partial hypotheses and expands them, approximating global sequence optimization. Sampling methods (temperature/top-k/top-p) trade some determinism for diversity and can improve open-ended generation quality.

| Method | Objective Style | Diversity | Compute Cost | Typical Use |
|---|---|---|---|---|
| Greedy | Local max at each token | Low | Low | Deterministic extraction / constrained outputs |
| Beam Search | Approx. sequence-level likelihood | Low-Medium | Medium-High | Translation/summarization style tasks |
| Sampling | Draw from controlled distribution | Medium-High | Low-Medium | Creative generation, ideation, conversational UX |

Interview tip: mention that beam search can produce repetitive or generic text in open-ended chat, so teams often prefer sampling for assistant-style interactions.

In [ ]:
import numpy as np

# Tiny transition model for beam-search demonstration
beam_vocab = ["A", "B", "C", "<END>"]
idx = {t: i for i, t in enumerate(beam_vocab)}
P = np.array([
    [0.05, 0.55, 0.35, 0.05],  # A -> ...
    [0.45, 0.05, 0.35, 0.15],  # B -> ...
    [0.35, 0.35, 0.05, 0.25],  # C -> ...
    [0.00, 0.00, 0.00, 1.00],  # END
])

def greedy_decode(start="A", steps=5):
    seq = [start]
    cur = start
    for _ in range(steps):
        nxt_id = int(np.argmax(P[idx[cur]]))
        cur = beam_vocab[nxt_id]
        seq.append(cur)
        if cur == "<END>":
            break
    return seq

def beam_search(start="A", steps=5, beam_width=3):
    beams = [([start], 0.0)]  # (sequence, logprob)
    for _ in range(steps):
        new_beams = []
        for seq, lp in beams:
            cur = seq[-1]
            if cur == "<END>":
                new_beams.append((seq, lp))
                continue
            probs = P[idx[cur]]
            for j, pj in enumerate(probs):
                if pj <= 0:
                    continue
                new_seq = seq + [beam_vocab[j]]
                new_lp = lp + np.log(pj)
                new_beams.append((new_seq, new_lp))
        new_beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        beams = new_beams
    return beams

print("Greedy:", greedy_decode())
print("Top beam hypotheses:")
for seq, lp in beam_search():
    print("  ", seq, "logp=", round(lp, 4))

---

## 13. Reproducibility and Seeds in Practice

A fixed random seed supports reproducibility only under stable conditions: same model version, tokenizer, prompt, decoding settings, and serving implementation. In distributed systems, tiny differences in kernel behavior or floating-point ordering can still produce divergent outputs over long generations.

For interview answers, distinguish *debug reproducibility* from *strict determinism*. Seeded sampling is excellent for regression tests and bug triage, but production systems should still track model version, prompt template version, and decoding config in logs for traceability.

---

## 14. Production Tuning Guide (Playbook)

A production-safe tuning sequence usually follows this order:

1. Start with low-variance decoding (low temperature, moderate top-p, bounded max tokens).
2. Validate factuality and task success on a representative eval set.
3. Increase diversity only if responses are too rigid.
4. Add repetition penalties only when loops/redundancy appear.
5. Enforce stop conditions and output schema checks.
6. Track latency and token cost regressions after every parameter change.

| Goal | Suggested Direction | Why |
|---|---|---|
| Reduce hallucinations | Lower temperature, tighten top-p, provide stronger context | Narrows probability mass around high-confidence tokens |
| Increase creativity | Raise temperature/top-p carefully | Explores broader token space |
| Reduce verbosity | Lower max_tokens + stronger stop conditions | Controls decode length and cost |
| Reduce repetition | Increase frequency/presence penalties gradually | Down-weights repeated token paths |

Interview-ready phrasing: "Tune for correctness first, then style; never optimize creativity before reliability constraints are met."

In [ ]:
---

## 15. Reliability Trend Intuition

As decoding explores larger portions of the token distribution, creative diversity can improve but factual reliability may degrade for knowledge-sensitive tasks. In interview discussions, describe this as an *entropy-reliability trade-off*: increasing exploration can uncover richer phrasing yet also raise the probability of low-support continuations.

A practical production stance is to keep entropy conservative for high-stakes workflows (support, legal, medical, policy), then selectively loosen parameters for ideation or brainstorming experiences.

---

## 15. Safety and Hallucination Control at Inference Time

Inference parameters do not "solve" hallucinations alone, but they materially influence failure frequency. Lower-entropy decoding often reduces fabricated details, especially when paired with grounding context and explicit uncertainty instructions. In interviews, emphasize layered mitigation: retrieval grounding, constrained output formats, post-generation verification, and conservative decoding for high-risk tasks.

A practical heuristic: for factual assistants, begin with lower temperature + moderate top-p + strict stop/schema validation, then loosen only if UX quality is too rigid.

---

## 16. Interview Focus (Detailed Q&A)

**Q1. What does temperature do mathematically?**  
It rescales logits before softmax, controlling distribution sharpness. Lower values make output more deterministic; higher values increase entropy and diversity.

**Q2. top-p vs top-k—how do you explain the difference?**  
top-k keeps a fixed number of candidates; top-p keeps a variable number based on cumulative probability mass. top-p adapts to uncertainty in the token distribution.

**Q3. How do you reduce hallucinations at inference time?**  
Use grounding context, lower entropy decoding, strict output constraints, and verification layers. Parameter tuning helps, but it must be combined with retrieval/evaluation guardrails.

**Q4. Why not always use greedy decoding?**  
Greedy can be brittle and repetitive in open-ended tasks. It is useful for deterministic extraction but often too rigid for conversational generation.

**Q5. When is beam search useful?**  
For tasks approximating sequence likelihood objectives (e.g., translation/summarization) where global scoring helps. It is less favored in creative chat settings.

**Q6. What is the role of max_tokens?**  
It is both a cost and latency control. It also constrains runaway generations and supports predictable response budgets.

**Q7. Presence penalty vs frequency penalty?**  
Presence penalty discourages reusing any seen token; frequency penalty scales with count and more strongly suppresses repeated loops.

**Q8. Are seeds enough for reproducibility?**  
They help, but reproducibility also requires fixed model version, tokenizer, prompt template, runtime stack, and decoding settings.

**Q9. What is your production tuning process?**  
Set reliability baseline first, tune one parameter at a time, evaluate on representative datasets, monitor cost/latency regressions, then gradually optimize style.

**Q10. What is the biggest anti-pattern in decoding tuning?**  
Tuning on a few demos and shipping without robust evaluation slices for difficult edge cases.

---

## 17. Summary for Interviews

Inference behavior is the product of both model capability and decoding policy. Temperature, top-k, top-p, penalties, and token limits are not isolated toggles; they jointly shape the probability landscape at every token step. Strong interview answers combine the mathematical view (logit transformation + truncation) with practical production discipline (reliability-first tuning, measurable evals, and observability).

## Key Takeaways (Interview-Ready)

- Temperature controls entropy; top-p/top-k control candidate truncation.
- Greedy, beam, and sampling each fit different task classes.
- Max token limits and stop conditions are critical for latency/cost safety.
- Hallucination mitigation at inference is multi-layered, not parameter-only.
- Reproducibility requires seeded sampling plus full config/version traceability.